# Cross-Environment Generalization

This notebook loads `data/generalization.xlsx`, restricts attention to the test rows, computes CIP and AR, and produces the tables used in the numerical-results subsection.


---
### Helper functions
---


In [1]:
from generalization_helpers import (
    POLICIES,
    LEARNED_POLICIES,
    INSTANCE_COLUMNS,
    DIST_GROUP_MAP,
    DIST_GROUP_ORDER,
    DIST_GROUP_LABELS,
    ensure_generalization_workbook,
    summarize,
    build_overall_table,
    build_policy_performance_table,
    assign_quantile_groups,
    build_group_panel,
    make_legend_handles,
    format_axis,
    to_latex,
    boxplot_on_axis,
)


---
### Pre-processing step
---


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rc("text", usetex=True)
plt.rc("font", family="serif")

Path("plots").mkdir(exist_ok=True)

ensure_generalization_workbook("data/final results.xlsx", "data/generalization.xlsx")

In [ ]:
path = "data/generalization.xlsx"
df = pd.read_excel(path, sheet_name="S5 merged")

# Keep only test rows and compute policy metrics used throughout the notebook.
df = df.loc[df["mode"].astype(str).str.lower().eq("test")].copy()

for policy in LEARNED_POLICIES:
    suffix = policy["suffix"]
    df[f"CIP_{suffix}"] = 100 * (1 - df[policy["cost_col"]] / df["avg_cost_basestock"])

df["CIP_BS"] = 0.0

rank_columns = [policy["cost_col"] for policy in POLICIES]
rank_df = df[rank_columns].rank(axis=1, method="average", ascending=True)
rank_df.columns = [f"AR_{policy['suffix']}" for policy in POLICIES]
df = pd.concat([df, rank_df], axis=1)

df["CV"] = df["demand_std"] / df["demand_mean"]
df["dist_group"] = df["distribution"].map(DIST_GROUP_MAP)


---
### Some Important Stats
---


In [ ]:
print("Number of test rows:", len(df))
print(
    "Number of distinct test instances:",
    df[INSTANCE_COLUMNS].drop_duplicates().shape[0],
)
print("Demand families:", df["distribution"].nunique())

family_instance_counts = (
    df[INSTANCE_COLUMNS]
    .drop_duplicates()
    .groupby("distribution")
    .size()
    .div(df["lead_time"].nunique() * df["cost_ratio"].nunique())
    .astype(int)
    .reset_index(name="num_mu_sigma_specs")
    .sort_values(["num_mu_sigma_specs", "distribution"], ascending=[False, True])
)

demand_specs = df[["distribution", "demand_mean", "demand_std"]].drop_duplicates().copy()
print("\nNumber of unique demand specifications (family, mu, sigma):", len(demand_specs))

print("\nDistinct (mu, sigma) specifications by family:")
print(family_instance_counts.to_string(index=False))


In [ ]:
demand_specs = df[["distribution", "demand_mean", "demand_std"]].drop_duplicates().copy()

for value_col, label, sort_cols in [
    ("demand_mean", "demand mean", ["distribution", "demand_std"]),
    ("demand_std", "demand std", ["distribution", "demand_mean"]),
]:
    min_value = demand_specs[value_col].min()
    max_value = demand_specs[value_col].max()

    print(f"Rows with minimum {label}:")
    print(
        demand_specs[demand_specs[value_col] == min_value]
        .sort_values(sort_cols)
        .to_string(index=False)
    )

    print(f"\nRows with maximum {label}:")
    print(
        demand_specs[demand_specs[value_col] == max_value]
        .sort_values(sort_cols)
        .to_string(index=False)
    )
    print()


In [ ]:
print("\nStats of coefficient of variation:")
print(df["CV"].describe())


---
### Overall generalization performance
---


In [ ]:
overall = build_overall_table(df)
overall

# print(to_latex(
#     overall,
#     index=False,
#     caption="Overall performance on the test rows.",
#     label="tab:overall_generalization",
# ))


---
### Performance across lead times
---


In [ ]:
lead_tbl = summarize(["lead_time"]).rename(columns={"lead_time": "L"})
lead_tbl

# print(to_latex(
#     lead_tbl,
#     index=False,
#     caption="Performance across lead times on the test rows.",
#     label="tab:lead_generalization",
# ))


In [ ]:
lead_times = sorted(df["lead_time"].dropna().unique())
lead_labels = [f"L={int(lead_time)}" for lead_time in lead_times]
lead_colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"][: len(lead_times)]
legend_handles = make_legend_handles(median_lw=0.5)

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)

for ax, policy in zip(axes, LEARNED_POLICIES):
    panel = build_group_panel(df, "lead_time", lead_times, f"CIP_{policy['suffix']}", labels=lead_labels)
    boxplot_on_axis(
        ax=ax,
        data=panel,
        cols=panel.columns.tolist(),
        box_colors=lead_colors,
        labels=lead_labels,
        title=policy["title"],
        xlabel="Lead time",
        ylabel=r"Cost improvement  $\%$" if ax is axes[0] else None,
        refline=0,
        vert=True,
        showfliers=True,
    )
    ax.set_ylim([-21, 155])
    format_axis(ax, tick_size=13, label_size=13, title_size=15)
    ax.legend(
        handles=legend_handles,
        loc="upper right",
        facecolor="#f2f2f2",
        frameon=True,
        fancybox=True,
        edgecolor="black",
        fontsize=11,
    )

fig.tight_layout()
fig.savefig("plots/leadtime_comparison.pdf", bbox_inches="tight")
plt.show()


In [ ]:
lead_perf_tbl = build_policy_performance_table(df, "lead_time", "L")
lead_perf_tbl

# print(
#     lead_perf_tbl.to_latex(
#         index=False,
#         caption="Mean CIP and AR across lead times.",
#         label="tab:leadtime_cip_ar",
#         escape=False,
#     )
# )


---
### Performance across cost ratios
---


In [ ]:
cost_tbl = summarize(["cost_ratio"]).rename(columns={"cost_ratio": "R"})
cost_tbl

# print(to_latex(
#     cost_tbl,
#     index=False,
#     caption="Performance across cost ratios on the test rows.",
#     label="tab:cost_generalization",
# ))


---
### Performance across demand distributions
---


In [ ]:
dist_tbl = summarize(["distribution"]).sort_values("distribution").reset_index(drop=True)
dist_tbl

# print(to_latex(
#     dist_tbl,
#     index=False,
#     caption="Performance across demand distributions on the test rows.",
#     label="tab:dist_generalization",
# ))


In [ ]:
group_colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
legend_handles = make_legend_handles(mean_size=8, outlier_size=9)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, policy in zip(axes, LEARNED_POLICIES):
    panel = build_group_panel(
        df,
        "dist_group",
        DIST_GROUP_ORDER,
        f"CIP_{policy['suffix']}",
        labels=DIST_GROUP_LABELS,
    )
    boxplot_on_axis(
        ax=ax,
        data=panel,
        cols=panel.columns.tolist(),
        box_colors=group_colors,
        labels=DIST_GROUP_LABELS,
        title=policy["title"],
        ylabel=r"Cost improvement $\%$" if ax is axes[0] else None,
        refline=0,
        vert=True,
        showfliers=True,
    )
    format_axis(ax, tick_size=15, label_size=15, title_size=15, rotation=35)
    ax.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=3,
        frameon=True,
        fancybox=False,
        edgecolor="black",
        facecolor="#f2f2f2",
        fontsize=10,
    )

fig.tight_layout()
fig.savefig("plots/dist_comparison.pdf", bbox_inches="tight")
plt.show()


In [ ]:
dist_order = (
    df.groupby("distribution")[["AR_P", "AR_E", "AR_N"]]
    .mean()
    .mean(axis=1)
    .sort_values(ascending=True)
    .index
    .tolist()
)

cmap = plt.get_cmap("tab20")
dist_colors = [cmap(i % 20) for i in range(len(dist_order))]
legend_handles = make_legend_handles(mean_size=7, outlier_size=8)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

for ax, policy in zip(axes, LEARNED_POLICIES):
    panel = build_group_panel(df, "distribution", dist_order, f"AR_{policy['suffix']}")
    boxplot_on_axis(
        ax=ax,
        data=panel,
        cols=panel.columns.tolist(),
        box_colors=dist_colors,
        labels=dist_order,
        title=policy["title"],
        xlabel="Distribution family",
        ylabel="Average rank" if ax is axes[0] else None,
        refline=None,
        vert=True,
        showfliers=True,
    )
    format_axis(ax, tick_size=11 if ax is not axes[0] else 11, label_size=12, title_size=14, rotation=60)
    ax.set_ylim(0.8, 4.2)

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    frameon=True,
    fancybox=False,
    edgecolor="black",
    bbox_to_anchor=(0.5, -0.04),
    fontsize=12,
)

fig.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()


In [ ]:
cv_labels = ["Low variability", "Moderate variability", "High variability"]
assign_quantile_groups(df, "CV", [1 / 3, 2 / 3], cv_labels, "cv_group_tercile")

policy_colors = ["tab:blue", "tab:orange", "tab:green"]
policy_labels = [policy["label"] for policy in LEARNED_POLICIES]
legend_handles = make_legend_handles(mean_size=7, outlier_size=8)

fig, axes = plt.subplots(1, 3, figsize=(14, 5.5), sharey=True)

for ax, cv_label in zip(axes, cv_labels):
    subset = df[df["cv_group_tercile"] == cv_label]
    panel = pd.DataFrame(
        {
            policy["label"]: subset[f"AR_{policy['suffix']}"] .reset_index(drop=True)
            for policy in LEARNED_POLICIES
        }
    )
    boxplot_on_axis(
        ax=ax,
        data=panel,
        cols=panel.columns.tolist(),
        box_colors=policy_colors,
        labels=policy_labels,
        title=cv_label,
        xlabel="Policy",
        ylabel="Average rank" if ax is axes[0] else None,
        refline=None,
        vert=True,
        showfliers=True,
    )
    format_axis(ax, tick_size=12, label_size=12, title_size=14)
    ax.set_ylim(0.8, 4.2)

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    frameon=True,
    fancybox=False,
    edgecolor="black",
    bbox_to_anchor=(0.5, -0.03),
    fontsize=12,
)

fig.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()


In [ ]:
cv_labels = [r"Low $\sigma/\mu$", r"Moderate $\sigma/\mu$", r"High $\sigma/\mu$"]
assign_quantile_groups(df, "CV", [0.25, 0.75], cv_labels, "cv_group_quartile")

df["lead_time_bucket"] = pd.cut(
    df["lead_time"],
    bins=[1.5, 4.0, 8.0],
    labels=[r"$L \in \{2,4\}$", r"$L \in \{6,8\}$"],
    ordered=True,
)

lead_time_bucket_labels = [r"$L \in \{2,4\}$", r"$L \in \{6,8\}$"]
policy_colors = ["tab:blue", "tab:orange", "tab:green"]
policy_labels = [policy["label"] for policy in LEARNED_POLICIES]
legend_handles = make_legend_handles(mean_size=7, outlier_size=8)

fig, axes = plt.subplots(
    len(lead_time_bucket_labels),
    len(cv_labels),
    figsize=(12, 5),
    sharex=True,
    sharey=True,
)

for row_idx, lead_bucket in enumerate(lead_time_bucket_labels):
    for col_idx, cv_label in enumerate(cv_labels):
        ax = axes[row_idx, col_idx]
        subset = df[
            (df["lead_time_bucket"] == lead_bucket)
            & (df["cv_group_quartile"] == cv_label)
        ]
        panel = pd.DataFrame(
            {
                policy["label"]: subset[f"AR_{policy['suffix']}"] .reset_index(drop=True)
                for policy in LEARNED_POLICIES
            }
        )
        boxplot_on_axis(
            ax=ax,
            data=panel,
            cols=panel.columns.tolist(),
            box_colors=policy_colors,
            labels=policy_labels,
            title=cv_label if row_idx == 0 else None,
            ylabel="Policy rank" if col_idx == 0 else None,
            refline=None,
            vert=True,
            showfliers=True,
        )
        format_axis(ax, tick_size=11, label_size=12, title_size=13)
        ax.set_ylim(0.8, 3.2)
        ax.set_yticks([1, 2, 3])
        ax.set_yticklabels([1, 2, 3])

fig.text(
    0.05,
    0.75,
    r"$L \in \{2,4\}$",
    rotation=0,
    va="center",
    ha="center",
    fontsize=13,
    bbox=dict(facecolor="lightgray", edgecolor="gray", boxstyle="round,pad=0.3"),
)
fig.text(
    0.05,
    0.34,
    r"$L \in \{6,8\}$",
    rotation=0,
    va="center",
    ha="center",
    fontsize=13,
    bbox=dict(facecolor="lightgray", edgecolor="gray", boxstyle="round,pad=0.3"),
)

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    frameon=True,
    fancybox=False,
    edgecolor="black",
    bbox_to_anchor=(0.5, -0.02),
    fontsize=12,
)

fig.tight_layout(rect=[0.09, 0.06, 1, 1])
fig.savefig("plots/AR_by_Lbucket_and_CV.pdf", bbox_inches="tight")
plt.show()
